In [ ]:
!cd /allah/stuff/freq/userdir/backtest_results
!ls

In [ ]:
lastest_json = '/allah/stuff/freq/userdir/backtest_results/backtest-result-2024-12-21_16-32-40.json'
import json
with open(lastest_json) as f:
    data = json.load(f)
data['strategy']['MultiTimeframeTEMAAgreement']['trades'][3]

In [1]:
# read backtest-result-2024-12-14_11-37-27.json
# backtest-result-2024-12-14_11-37-27.meta.json
# backtest-result-2024-12-14_11-37-27_market_change.feather

import json
import pandas as pd
with open('/allah/stuff/freq/userdir/backtest_results/backtest-result-2024-12-24_03-00-49.json', 'r') as f:
    long_backtest_results = json.load(f)
# Load the backtest results
with open('/allah/stuff/freq/userdir/backtest_results/backtest-result-2024-12-24_02-55-57.json', 'r') as f:
    short_backtest_results = json.load(f)

short_trades = short_backtest_results["strategy"]["TEMA50Trailing_All_Short_Strategy"]["trades"]
long_trades = long_backtest_results["strategy"]["TEMA50Trailing_All_Long_Strategy"]["trades"]

df_trades_long = pd.DataFrame(long_trades)
df_trades_short = pd.DataFrame(short_trades)



In [ ]:
df_trades_long

In [2]:
import pandas as pd
import numpy as np

# Assuming df_trades_long and df_trades_short are already loaded with your data

# Create profitability columns for long and short trades
df_trades_long['long_profitability'] = df_trades_long['profit_abs']
df_trades_short['short_profitability'] = df_trades_short['profit_abs']

# Include `enter_tag` in the merge to trace the entry condition
df_combined = pd.merge(
    df_trades_long[['open_date', 'long_profitability', 'enter_tag']],  # Include enter_tag from long trades
    df_trades_short[['open_date', 'short_profitability']],
    on='open_date', 
    how='left'
)

# Add a new column to classify trades based on profitability comparison
df_combined['type'] = np.where(
    df_combined['long_profitability'] > df_combined['short_profitability'], 
    'LONG_WIN',  # Long profit is greater
    'SHORT_WIN'  # Otherwise, Short profit is greater
)

# Optional: Convert the type column to binary for ML (e.g., 1 for LONG_WIN, 0 for SHORT_WIN)
df_combined['type_binary'] = np.where(df_combined['type'] == 'LONG_WIN', 1, 0)

# Save the resulting DataFrame to a Parquet file
df_combined.to_parquet("/allah/data/parquet/final_df.parquet")

# Print a summary of the classification
long_win_count = (df_combined['type'] == 'LONG_WIN').sum()
short_win_count = (df_combined['type'] == 'SHORT_WIN').sum()

print(f"LONG_WIN Count: {long_win_count}")
print(f"SHORT_WIN Count: {short_win_count}")

# Optional: Analyze the distribution of enter_tag for LONG_WIN vs. SHORT_WIN
long_win_tags = df_combined[df_combined['type'] == 'LONG_WIN']['enter_tag'].value_counts()
short_win_tags = df_combined[df_combined['type'] == 'SHORT_WIN']['enter_tag'].value_counts()

print("\nLONG_WIN Enter Tags Distribution:")
print(long_win_tags)
print("\nSHORT_WIN Enter Tags Distribution:")
print(short_win_tags)


LONG_WIN Count: 11574
SHORT_WIN Count: 11833

LONG_WIN Enter Tags Distribution:
enter_tag
tema50_up      5806
tema50_down    5768
Name: count, dtype: int64

SHORT_WIN Enter Tags Distribution:
enter_tag
tema50_down    5936
tema50_up      5897
Name: count, dtype: int64


In [4]:
df_combined.tail(30)

,open_date,long_profitability,enter_tag,short_profitability,type,type_binary
23377,2024-12-23 12:12:00+00:00,0.007886,tema50_down,-0.581584,LONG_WIN,1
23378,2024-12-23 12:21:00+00:00,-0.263087,tema50_up,0.426712,SHORT_WIN,0
23379,2024-12-23 12:57:00+00:00,-0.452663,tema50_down,0.388402,SHORT_WIN,0
23380,2024-12-23 13:00:00+00:00,-0.482573,tema50_up,0.468427,SHORT_WIN,0
23381,2024-12-23 13:03:00+00:00,-0.454389,tema50_down,0.413367,SHORT_WIN,0
23382,2024-12-23 07:18:00+00:00,1.385360,tema50_down,-0.819305,LONG_WIN,1
23383,2024-12-23 07:27:00+00:00,1.043711,tema50_up,-0.670004,LONG_WIN,1
23384,2024-12-23 07:12:00+00:00,1.030952,tema50_up,-0.601082,LONG_WIN,1
23385,2024-12-23 14:06:00+00:00,-0.648247,tema50_up,0.843707,SHORT_WIN,0
23386,2024-12-23 14:36:00+00:00,-0.944295,tema50_down,0.622801,SHORT_WIN,0


In [18]:
import pandas as pd

# Assuming df_trades_long and df_trades_short are already loaded with your data

# Fixing the typo in column names for clarity
df_trades_long['is_long_profit'] = df_trades_long['profit_abs'] > 0
df_trades_long['long_profitability'] = df_trades_long['profit_abs']

df_trades_short['is_short_profit'] = df_trades_short['profit_abs'] > 0
df_trades_short['short_profitability'] = df_trades_short['profit_abs']

# Merging the DataFrames based on a common column 'open_date' if it's unique or an appropriate key
# Ensure that 'open_date' is a suitable key for merging; if not, adjust accordingly
df_combined = pd.merge(df_trades_long, df_trades_short[['open_date', 'is_short_profit', 'short_profitability']],
                       on='open_date', how='left')

# Selecting only the required columns for the final DataFrame
final_df = df_combined[['open_date', 'is_long_profit', 'long_profitability', 'is_short_profit', 'short_profitability', 'enter_tag']]


In [ ]:
final_df

In [ ]:
final_df['open_date'] = pd.to_datetime(final_df['open_date'], utc=True)

final_df['target_date'] = final_df['open_date']  - pd.Timedelta(minutes=3)

In [ ]:
final_df

In [ ]:
final_df['target_date']

In [ ]:
print(final_df['open_date'].dtype)

In [ ]:
final_df

In [ ]:
# Filtering the DataFrame for rows where is_long_profit is True and is_short_profit is False
filtered_df = final_df[(final_df['type'] == 'LONG_WIN') & (final_df['enter_tag'] == 'tema50_up')]
# Filtering the DataFrame for rows where is_long_profit is True and is_short_profit is False
filtered_df.head(20)

final_df.type.value_counts()



In [ ]:
# Assuming final_df is loaded with your data
# Calculate the counts for each combination
count_both_false = len(final_df[(final_df['is_long_profit'] == False) & (final_df['is_short_profit'] == False)])
count_both_true = len(final_df[(final_df['is_long_profit'] == True) & (final_df['is_short_profit'] == True)])
count_long_true_short_false = len(final_df[(final_df['is_long_profit'] == True) & (final_df['is_short_profit'] == False)])
count_long_false_short_true = len(final_df[(final_df['is_long_profit'] == False) & (final_df['is_short_profit'] == True)])

# Print out the results
print("Both long and short profit False:", count_both_false)
print("Both long and short profit True:", count_both_true)
print("Long profit True and short profit False:", count_long_true_short_false)
print("Long profit False and short profit True:", count_long_false_short_true)

import numpy as np

# Define conditions
conditions = [
    (final_df['is_long_profit'] == False) & (final_df['is_short_profit'] == False), # Both false
    (final_df['is_long_profit'] == True) & (final_df['is_short_profit'] == True),   # Both true
    (final_df['is_long_profit'] == True) & (final_df['is_short_profit'] == False),  # Long win only
    (final_df['is_long_profit'] == False) & (final_df['is_short_profit'] == True)   # Short win only
]

# Define choices corresponding to the conditions
choices = [
    'LOSE',     # Both lose
    'WIN',      # Both win
    'LONG_WIN', # Only long win
    'SHORT_WIN' # Only short win
]

# Create a new column 'type' with values based on conditions
final_df['type'] = np.select(conditions, choices, default='undefined')

# Print or view the DataFrame to confirm the new column is added correctly
final_df.to_parquet("/allah/data/parquet/final_df.parquet")


In [ ]:
df_trades_long['long_profitability']

In [ ]:
df_trades_short[['enter_tag', 'is_short', 'profit_abs', 'open_date']]

In [ ]:
import asyncio
import ccxt.pro
# run async in jpynb
import nest_asyncio
nest_asyncio.apply()


async def fetch_trades():
    # Initialize Binance exchange
    exchange = ccxt.pro.binance({
        "enableRateLimit": True,  # Enable rate limit to avoid getting blocked
    })

    symbol = "ETH/USDT:USDT"  # Trading pair

    try:
        # Subscribe to the WebSocket feed for trades
        while True:
            trade = await exchange.watch_trades(symbol)
            for t in trade:
                print({
                    "timestamp": t["timestamp"],    # Timestamp of the trade
                    "datetime": t["datetime"],      # Human-readable date
                    "symbol": t["symbol"],          # Trading pair
                    "price": t["price"],            # Trade price
                    "amount": t["amount"],          # Trade amount
                    "side": t["side"],              # Buy or Sell
                })
    except Exception as e:
        print(f"Error: {e}")
    finally:
        await exchange.close()  # Close the WebSocket connection when done

# Run the asyncio loop
asyncio.run(fetch_trades())
